import libraries(I provide all libs that I need when make this tasks, if you need some external import them here)

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, NullType
from pyspark.sql.functions import col
from pyspark.sql.functions import max, avg, min
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import when

In [3]:
import findspark
findspark.init()

In [4]:
import os
import sys

# Replace the path below with the output from Step 1
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"

# Force Python to find the Java executables
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

create local SparkSession

In [5]:
spark = SparkSession.builder \
    .appName("ExamScoreAnalysis") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/08 13:51:42 WARN Utils: Your hostname, Andrews-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.132.33.244 instead (on interface en0)
26/01/08 13:51:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/08 13:51:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


read csv with inferschema

In [7]:
%%time
df = (spark.read.format("csv")
      .option("header", "True")
      .option("inferSchema", "True")
      .load("ds_salaries.csv"))

CPU times: user 3.27 ms, sys: 1.97 ms, total: 5.25 ms
Wall time: 1.28 s


read csv one more time with the same code and you will see that it almostly don't take time, because info already in SparkSession and it will not read nothing
from this file

In [8]:
%%time
df1 = (spark.read.format("csv")
      .option("header", "True")
      .option("inferSchema", "True")
      .load("exam_score_prediction.csv"))

CPU times: user 1.75 ms, sys: 1.82 ms, total: 3.58 ms
Wall time: 25.7 ms


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/silvia/work/spark_demo_course/1_PySpark_Basics/exam_score_prediction.csv. SQLSTATE: 42K03

write schema of scv on screen

In [9]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- work_year: integer (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- salary_currency: string (nullable = true)
 |-- salary_in_usd: integer (nullable = true)
 |-- employee_residence: string (nullable = true)
 |-- remote_ratio: integer (nullable = true)
 |-- company_location: string (nullable = true)
 |-- company_size: string (nullable = true)



create schema of this scv

In [11]:
%%time
schema = StructType([
    StructField("index", IntegerType(), True),
    StructField("work_year", IntegerType(), True),
    StructField("experience_level", StringType(), True),
    StructField("employment_type", StringType(), True),
    StructField("job_title", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("salary_currency", StringType(), True),
    StructField("salary_in_usd", IntegerType(), True),
    StructField("employee_residence", StringType(), True),
    StructField("remote_ratio", IntegerType(), True),
    StructField("company_location", StringType(), True),
    StructField("company_size", StringType(), True)
])

CPU times: user 30 μs, sys: 3 μs, total: 33 μs
Wall time: 34.1 μs


restart kernel without cleaning output and after restarting you need to initialize SparkSession, after initialize start execute only cells from cell with schema=
=StructType.... 
To restart kernel click Kernel, Restart.

read ds_salaries with predefined schema and compare results from this cell and cell with inferSchema

In [17]:
df_schema = (spark.read.format("csv")
      .option("header", "True")
      .schema(schema)
      .load("ds_salaries.csv"))

df_schema.show(5)

+-----+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|index|work_year|experience_level|employment_type|           job_title|salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+-----+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|    0|     2020|              MI|             FT|      Data Scientist| 70000|            EUR|        79833|                DE|           0|              DE|           L|
|    1|     2020|              SE|             FT|Machine Learning ...|260000|            USD|       260000|                JP|           0|              JP|           S|
|    2|     2020|              SE|             FT|   Big Data Engineer| 85000|            GBP|       109024|                GB|          50|     

this happens because read operation is lazy(transformation), but if you use inferschema it start to be action that will create Spark Job, because Spark need to loop throw all file to check datatypes for all columns and this can harm to your code(if we compare to parquet, it will also go to check data types, but parquet provide meta information, so Spark will not go throw all file, he will just read meta information, but csv don't provide such meta information). Also header make Spark to create one more Spark Job to check first line
to define name of columns and remember to skeep it when reading. Actual reading start when you will use first action. More about Spark Jobs you will see in next topic

write schema of scv on screen one more time and compare with previous

In [15]:
df_schema.printSchema()

root
 |-- index: integer (nullable = true)
 |-- work_year: integer (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- salary_currency: string (nullable = true)
 |-- salary_in_usd: integer (nullable = true)
 |-- employee_residence: string (nullable = true)
 |-- remote_ratio: integer (nullable = true)
 |-- company_location: string (nullable = true)
 |-- company_size: string (nullable = true)



now continue to work with one of the dataframes that you create

print data in dataframe using df.show

In [16]:
df.show(5)

+---+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|_c0|work_year|experience_level|employment_type|           job_title|salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+---+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|  0|     2020|              MI|             FT|      Data Scientist| 70000|            EUR|        79833|                DE|           0|              DE|           L|
|  1|     2020|              SE|             FT|Machine Learning ...|260000|            USD|       260000|                JP|           0|              JP|           S|
|  2|     2020|              SE|             FT|   Big Data Engineer| 85000|            GBP|       109024|                GB|          50|              GB|

print data in dataframe using display(df.toPandas())

In [19]:
display(df.toPandas().head(5))

,_c0,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M
3,3,2020,MI,FT,Product Data Analyst,20000,USD,20000,HN,0,HN,S
4,4,2020,SE,FT,Machine Learning Engineer,150000,USD,150000,US,50,US,L


create df_job_title that consists from all job_titles without duplicates

In [22]:
df_job_title = df.select("job_title").distinct()

print all rows from df_job_titles without truncating jobs

In [24]:
df_job_title.show(truncate=False)

+----------------------------------------+
|job_title                               |
+----------------------------------------+
|3D Computer Vision Researcher           |
|Lead Data Engineer                      |
|Head of Machine Learning                |
|Data Specialist                         |
|Data Analytics Lead                     |
|Machine Learning Scientist              |
|Lead Data Analyst                       |
|Data Engineering Manager                |
|Staff Data Scientist                    |
|ETL Developer                           |
|Director of Data Engineering            |
|Product Data Analyst                    |
|Principal Data Scientist                |
|AI Scientist                            |
|Director of Data Science                |
|Machine Learning Engineer               |
|Lead Data Scientist                     |
|Machine Learning Infrastructure Engineer|
|Data Science Engineer                   |
|Machine Learning Manager                |
+----------

create  df_analytic that will consists from max, avg, min USD salaries for all job_titles using groupBy. name of fields is avg_salary, min_salary, max_salary

In [33]:

df_analytic = df.groupBy("job_title").agg(
    avg("salary_in_usd").alias("avg_salary"),
    min("salary_in_usd").alias("min_salary"),
    max("salary_in_usd").alias("max_salary")
)

print all rows from df_analytic without trancating jobs

In [32]:
df_analytic.show(truncate=False)

+----------------------------------------+------------------+----------+----------+
|job_title                               |avg_salary        |min_salary|max_salary|
+----------------------------------------+------------------+----------+----------+
|3D Computer Vision Researcher           |5409.0            |5409      |5409      |
|Lead Data Engineer                      |139724.5          |56000     |276000    |
|Head of Machine Learning                |79039.0           |79039     |79039     |
|Data Specialist                         |165000.0          |165000    |165000    |
|Data Analytics Lead                     |405000.0          |405000    |405000    |
|Machine Learning Scientist              |158412.5          |12000     |260000    |
|Lead Data Analyst                       |92203.0           |19609     |170000    |
|Data Engineering Manager                |123227.2          |59303     |174000    |
|Staff Data Scientist                    |105000.0          |105000    |1050

now you need to add in df_analytic column row_id, that will show order of all job_titles depending on avg salary. they should be descending

In [40]:
windowSpecAgg = Window.orderBy("avg_salary")

df_analytic = df_analytic.withColumn(
    "row_id", row_number().over(windowSpecAgg)
)

print all data from df_analytic

In [41]:
df_analytic.show()

+--------------------+------------------+----------+----------+------+
|           job_title|        avg_salary|min_salary|max_salary|row_id|
+--------------------+------------------+----------+----------+------+
|3D Computer Visio...|            5409.0|      5409|      5409|     1|
|Product Data Analyst|           13036.0|      6072|     20000|     2|
|        NLP Engineer|           37236.0|     37236|     37236|     3|
|Computer Vision E...|44419.333333333336|     10000|    125000|     4|
|   Big Data Engineer|           51974.0|      5882|    114047|     5|
|       ETL Developer|           54957.0|     54957|     54957|     6|
|Finance Data Analyst|           61896.0|     61896|     61896|     7|
|Data Analytics En...|          64799.25|     20000|    110000|     8|
|        AI Scientist| 66135.57142857143|     12000|    200000|     9|
|Data Science Cons...| 69420.71428571429|      5707|    103000|    10|
|     BI Data Analyst| 74755.16666666667|      9272|    150000|    11|
|Data 

it isn't beautifull, so we need to put now row_id on first place in df_analytic

In [43]:
df_analytic = df_analytic.select("row_id", "job_title", "avg_salary", "min_salary", "max_salary")

print df_analytic now

In [45]:
df_analytic.show(truncate=False)

+------+------------------------------+------------------+----------+----------+
|row_id|job_title                     |avg_salary        |min_salary|max_salary|
+------+------------------------------+------------------+----------+----------+
|1     |3D Computer Vision Researcher |5409.0            |5409      |5409      |
|2     |Product Data Analyst          |13036.0           |6072      |20000     |
|3     |NLP Engineer                  |37236.0           |37236     |37236     |
|4     |Computer Vision Engineer      |44419.333333333336|10000     |125000    |
|5     |Big Data Engineer             |51974.0           |5882      |114047    |
|6     |ETL Developer                 |54957.0           |54957     |54957     |
|7     |Finance Data Analyst          |61896.0           |61896     |61896     |
|8     |Data Analytics Engineer       |64799.25          |20000     |110000    |
|9     |AI Scientist                  |66135.57142857143 |12000     |200000    |
|10    |Data Science Consult

here you need to create df_exp_lvl with the biggest usd_salary(biggest_salary) for each experience_level(you need to save all fields like in entire dataframe)

In [53]:
windowSpecAgg = Window.partitionBy("experience_level").orderBy(col("salary_in_usd").desc())

df_exp_lvl = (df_schema.withColumn("rank", row_number().over(windowSpecAgg))
              .filter(col("rank") == 1)
              .drop("rank")
              .withColumnRenamed("salary_in_usd", "biggest_salary"))


print here df_exp_lvl

In [54]:
df_exp_lvl.show()

+-----+---------+----------------+---------------+--------------------+------+---------------+--------------+------------------+------------+----------------+------------+
|index|work_year|experience_level|employment_type|           job_title|salary|salary_currency|biggest_salary|employee_residence|remote_ratio|company_location|company_size|
+-----+---------+----------------+---------------+--------------------+------+---------------+--------------+------------------+------------+----------------+------------+
|   37|     2020|              EN|             FT|Machine Learning ...|250000|            USD|        250000|                US|          50|              US|           L|
|  252|     2021|              EX|             FT|Principal Data En...|600000|            USD|        600000|                US|         100|              US|           L|
|   33|     2020|              MI|             FT|  Research Scientist|450000|            USD|        450000|                US|           0

create df_best that consists from rows where salary of guy same as biggest salary for other people in his exp_lvl and choose only columns: id, experience_level, biggest_salary, employee_residence

In [56]:
windowSpecAgg = Window.partitionBy("experience_level")

df_best = (df_schema.withColumn("biggest_salary", max("salary_in_usd").over(windowSpecAgg))
           .filter(col("salary_in_usd") == col("biggest_salary"))
           .select("index", "experience_level", "biggest_salary", "employee_residence"))

print df_best

In [57]:
df_best.show()

+-----+----------------+--------------+------------------+
|index|experience_level|biggest_salary|employee_residence|
+-----+----------------+--------------+------------------+
|   37|              EN|        250000|                US|
|  252|              EX|        600000|                US|
|   33|              MI|        450000|                US|
|   97|              MI|        450000|                US|
|   63|              SE|        412000|                US|
+-----+----------------+--------------+------------------+



drop duplicates if exist by experience_level

In [64]:
df_best = df_best.dropDuplicates(["experience_level"])

print df_best

In [65]:
df_best.show()

+-----+----------------+--------------+------------------+
|index|experience_level|biggest_salary|employee_residence|
+-----+----------------+--------------+------------------+
|   37|              EN|        250000|                US|
|  252|              EX|        600000|                US|
|   33|              MI|        450000|                US|
|   63|              SE|        412000|                US|
+-----+----------------+--------------+------------------+



create df_new_best from df_best without id, and make the next: when exp_level = MI we want middle, when SE we want senior, else Null

In [75]:
df_new_best = df_best.withColumn("experience_level", when(col("experience_level") == "MI", "middle")
                                                     .when(col("experience_level") == "SE", "senior"))

print df_new_best

In [76]:
df_new_best.show()

+-----+----------------+--------------+------------------+
|index|experience_level|biggest_salary|employee_residence|
+-----+----------------+--------------+------------------+
|   37|            NULL|        250000|                US|
|  252|            NULL|        600000|                US|
|   33|          middle|        450000|                US|
|   63|          senior|        412000|                US|
+-----+----------------+--------------+------------------+



write df_new_best like 1.csv and load then it to df_final

In [79]:
df_new_best.write.options(header='True', delimiter=',').mode('overwrite').csv("1.csv")

df_final = spark.read.format("csv")     \
            .option("header", "True")     \
            .option("inferSchema", "True") \
            .load("1.csv")

print df_final

In [80]:
df_final.show()

+-----+----------------+--------------+------------------+
|index|experience_level|biggest_salary|employee_residence|
+-----+----------------+--------------+------------------+
|   37|            NULL|        250000|                US|
|  252|            NULL|        600000|                US|
|   33|          middle|        450000|                US|
|   63|          senior|        412000|                US|
+-----+----------------+--------------+------------------+



filter df_final to delete experience_level where it Null, then join this table by biggest_salary(salary_in_usd) and employee_residence with entire df

In [83]:
df_final = df_final.filter(col("experience_level").isNotNull())
df_final = df_schema.join(df_final,
                          (df_schema.salary_in_usd == df_final.biggest_salary) &
                          (df_schema.employee_residence == df_final.employee_residence),
                          "inner")

print df_final

In [84]:
df_final.show()

+-----+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+-----+----------------+--------------+------------------+
|index|work_year|experience_level|employment_type|           job_title|salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|index|experience_level|biggest_salary|employee_residence|
+-----+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+-----+----------------+--------------+------------------+
|   33|     2020|              MI|             FT|  Research Scientist|450000|            USD|       450000|                US|           0|              US|           M|   33|          middle|        450000|                US|
|   63|     2020|              SE|             FT|      Data Scientist|412000|          

last task is to save in variable and then print this variable of the biggest salary_in_usd from df_final

In [87]:
biggest_sal_val = df_final.select(max("biggest_salary")).collect()
biggest_sal_val[0][0]

450000

It is the end of PySpark basics. In other lessons you will learn optimizations technics and how to make distributed system